# Planificador geoespacial de eclipses

Catálogo de los próximos tres años y consulta de datos preparados. El usuario elige una fecha real NASA, selecciona un país y revisa zonas ordenadas, circunstancias y climatología. La interfaz en `app/index.html` incorpora un mapa local interactivo; alojamientos y precios se incorporarán posteriormente.


In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/eclipse_pipeline.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src import eclipse_catalogs as catalogs, eclipse_product as product
AS_OF = None  # la ventana empieza hoy (UTC)
FUTURE_YEARS = 3
EVENT_DATE = "2027-08-02"  # ejemplo; si ya ha pasado se usa el primer eclipse de la ventana
OFFLINE = True


## Fechas disponibles

El horizonte de tres años limita los eclipses del producto. El catálogo solo necesita los siglos que contienen esa ventana. El histórico climático es independiente y utiliza 30 años disponibles por defecto. Este notebook consulta copias locales.

In [2]:
planning_start, planning_end = catalogs.planning_window(AS_OF, FUTURE_YEARS)
events = catalogs.load_planning_catalog(as_of=AS_OF, future_years=FUTURE_YEARS, offline=OFFLINE)
print("Ventana:", planning_start, "hasta", planning_end, "(fin excluido)")
print("Eventos:", len(events))
print(events[["event_date", "event_family", "eclipse_type"]].to_string(index=False))
if EVENT_DATE not in set(events.event_date):
    EVENT_DATE = events.event_date.iloc[0]
event = catalogs.select_event(events, EVENT_DATE)


Ventana: 2026-10-24 hasta 2029-10-24 (fin excluido)
Eventos: 14
event_date event_family eclipse_type
2027-02-06        solar      annular
2027-02-20        lunar    penumbral
2027-07-18        lunar    penumbral
2027-08-02        solar        total
2027-08-17        lunar    penumbral
2028-01-12        lunar      partial
2028-01-26        solar      annular
2028-07-06        lunar      partial
2028-07-22        solar        total
2028-12-31        lunar        total
2029-01-14        solar      partial
2029-06-12        solar      partial
2029-06-26        lunar        total
2029-07-11        solar      partial


## Selección de país y circunstancias

Se combinan los destinos astronómicos de Silver con sus resultados climáticos Gold ya preparados, sin descargar ni recalcular astronomía o clima. En solares, el país se ordena por la mayor superficie del disco ocultada entre sus puntos analizados y se muestran países y zonas con al menos 70 %. No describe el porcentaje del territorio ni garantiza el máximo de todo el país. Para lunares se utiliza la altura en el máximo global y la duración global de fase.


In [3]:
payload = product.build_planner_data(events, planning_start, planning_end, ROOT)
selected = next(item for item in payload["events"] if item["event_id"] == event["event_id"])
print(selected["label"], "Estado:", selected["status"])
locations = pd.DataFrame(selected["locations"])
countries = pd.DataFrame(selected["countries"])
COUNTRY_CODE = "ES"
metric = "obscuration_pct" if event["event_family"] == "solar" else "body_altitude_deg"
country_metric = "max_obscuration_pct" if event["event_family"] == "solar" else "max_body_altitude_deg"
if not countries.empty:
    if event["event_family"] == "solar":
        countries = countries[countries["max_obscuration_pct"].ge(70)]
    print("Países disponibles:")
    print(countries[["country_code", "country_name", country_metric, "location_count", "climate_location_count"]].to_string(index=False))
if not locations.empty:
    visible = locations[locations["country_code"].eq(COUNTRY_CODE)].sort_values(metric, ascending=False)
    if event["event_family"] == "solar":
        visible = visible[visible["obscuration_pct"].ge(70)]
    print("Zonas del país seleccionado:", COUNTRY_CODE)
    print(visible[["name", "local_phase", "peak_time_utc", metric, "phase_duration_s", "duration_scope", "climate_status"]].to_string(index=False))
else:
    print("Preparar el evento con el pipeline antes de consultar sus ubicaciones.")


02-08-2027 · Eclipse solar total Estado: ready
Países disponibles:
country_code      country_name  max_obscuration_pct  location_count  climate_location_count
          EG            Egipto           100.000000               7                       1
          LY             Libia           100.000000               2                       1
          YE             Yemen           100.000000               2                       1
          TN             Túnez           100.000000               2                       1
          SA    Arabia Saudita           100.000000               3                       1
          DZ           Argelia           100.000000               4                       1
          MA         Marruecos           100.000000               5                       1
          ES            España            98.369597               7                       1
          MT             Malta            97.282551               1                       1
          PT 

## Ventana meteorológica

Los archivos horarios conservan 19 variables y los días relativos. Los resúmenes a la hora del máximo y los acumulados de todo el día son salidas diferentes. Se indica la cobertura real de la preparación, sin suponer que la muestra ya tenga 30 años.

In [4]:
gold = ROOT / "data/gold/events" / EVENT_DATE
silver = ROOT / "data/silver/events" / EVENT_DATE
report = json.loads((gold / "pipeline_quality_report.json").read_text(encoding="utf-8"))
hourly_path = silver / "climate_hourly.csv"
# El archivo horario completo no se versiona: se regenera con el pipeline.
hourly = pd.read_csv(hourly_path) if hourly_path.exists() else pd.DataFrame(columns=["day_offset"])
samples = pd.read_csv(gold / "climate_event_samples.csv")
daily = pd.read_csv(gold / "climate_daily_conditions.csv")
print("Años históricos preparados:", len(report["climate"]["years"]))
print("Filas horarias:", len(hourly) if hourly_path.exists() else "no disponibles; ejecutar python src/eclipse_pipeline.py", "Muestras a la hora objetivo:", len(samples))
print(hourly.groupby("day_offset").size().to_string())
print(daily.head(5).to_string(index=False))


Años históricos preparados: 30
Filas horarias: 43200 Muestras a la hora objetivo: 1800
day_offset
-2    8640
-1    8640
 0    8640
 1    8640
 2    8640
              name  reference_year  day_offset      event_id event_date  mean_temperature_c  min_temperature_c  max_temperature_c  mean_cloud_cover_pct  max_wind_speed_kmh  rain_mm  precipitation_mm  valid_rain_hours  valid_temperature_hours
Al-Mansurah, EGYPT            1997          -2 NASA-SE-09568 2027-08-02           31.895833               26.4               37.1              0.000000                21.9      0.0               0.0                24                       24
Al-Mansurah, EGYPT            1997          -1 NASA-SE-09568 2027-08-02           31.816667               25.7               37.2              0.291667                20.3      0.0               0.0                24                       24
Al-Mansurah, EGYPT            1997           0 NASA-SE-09568 2027-08-02           32.591667               25.9           

## Comparación de destinos con climatología

Las frecuencias históricas de cielo despejado y lluvia no son pronósticos. Una cobertura inferior a 15 años se identifica como provisional. La lista astronómica puede contener más zonas que la muestra climática: las demás conservan climatología pendiente y valores nulos. Las referencias astronómicas que pueden caer en el mar se excluyen del selector.


In [5]:
if not locations.empty:
    prepared = locations[locations["climate_status"].isin(["ready", "partial"])]
    columns = ["rank", "name", "country_name", "clear_probability_pct", "rain_probability_pct", "mean_cloud_cover_pct", "valid_cloud_years", "valid_rain_years", "ranking_provisional"]
    print("Destinos con datos climáticos:", len(prepared), "de", len(locations))
    print(prepared[columns].head(12).to_string(index=False))
print("Estado de la preparación del catálogo:", payload["counts"])


Destinos con datos climáticos: 9 de 300
 rank                 name   country_name  clear_probability_pct  rain_probability_pct  mean_cloud_cover_pct  valid_cloud_years  valid_rain_years  ranking_provisional
    1   Al-Mansurah, EGYPT         Egipto              93.333333              0.000000              2.433333               30.0              30.0                False
    2      Banghazi, LIBYA          Libia              90.000000              0.000000              5.100000               30.0              30.0                False
    3        Sfax, TUNISIA          Túnez              90.000000              0.000000              4.933333               30.0              30.0                False
    4 Jiddah, SAUDI ARABIA Arabia Saudita              63.333333              0.000000             24.266667               30.0              30.0                False
    5        Oran, ALGERIA        Argelia              70.000000              3.333333             17.600000               30

## Dataset para Machine Learning: solo información disponible al planificar

Las etiquetas (`clear_sky_proxy`, `rain_proxy`) se calculan con la nubosidad y la lluvia de la hora del eclipse, así que esas variables del mismo año no pueden ser predictores. `planning_ml_dataset.csv` tiene una fila por ubicación y año: predictores conocidos al planificar (ubicación, fecha, hora solar, altura del astro y climatología de años **anteriores**), la frecuencia histórica previa como línea base y la etiqueta de ese año. La fila `query` es la del propio eclipse, sin etiqueta. La validación es temporal por año.

In [6]:
ml = pd.read_csv(gold / "planning_ml_dataset.csv")
roles = report["ml_dataset"]["columns"]
print("Predictores:", ", ".join(roles["predictors"]))
print("Filas:", ml.row_role.value_counts().to_dict())
history = ml[ml.row_role.eq("history") & ml.prior_years.ge(report["ml_dataset"]["min_prior_years_for_evaluation"])]
brier = ((history.baseline_clear_probability - history.clear_sky_proxy) ** 2).mean()
print(f"Línea base (frecuencia previa), Brier en {len(history)} filas evaluables: {brier:.3f}")
print(ml[ml.row_role.eq("query")][["name", "prior_years", "prior_clear_frequency", "local_solar_hour"]].to_string(index=False))

Predictores: latitude, longitude, elevation_m, grid_latitude, grid_longitude, day_of_year, local_solar_hour, body_altitude_deg, prior_years, prior_clear_frequency, prior_rain_frequency, prior_window_clear_frequency, prior_mean_cloud_cover_pct, prior_mean_cloud_cover_low_pct
Filas: {'history': 360, 'query': 12}
Línea base (frecuencia previa), Brier en 240 filas evaluables: 0.132
                     name  prior_years  prior_clear_frequency  local_solar_hour
       Al-Mansurah, EGYPT           30               0.933333            12.153
          Banghazi, LIBYA           30               0.900000            10.838
     Jiddah, SAUDI ARABIA           30               0.633333            13.030
            Oran, ALGERIA           30               0.700000             8.836
Referencia central NASA 3           30               0.833333            10.420
Referencia central NASA 4           30               0.966667            11.654
Referencia central NASA 5           30               0.6666

## Consulta local y preparación inicial

Para preparar la ventana completa una vez: `python src/eclipse_pipeline.py`. Después se abre `app/index.html`: seleccionar fechas, países o puntos del mapa no realiza peticiones externas. `python src/eclipse_pipeline.py --export-only` reconstruye el selector desde Silver y Gold y empaqueta la cartografía local. `--refresh` actualiza explícitamente las fuentes. La reutilización normal exige resultados completos y configuración compatible. Los eventos pendientes o parciales y las zonas sin climatología muestran su estado.


In [7]:
print("Pronóstico guardado:", selected["forecast_status"]["status"])
print("Selector:", ROOT / "app/index.html")
print("Catálogo:", ROOT / "data/gold/planning_eclipse_catalog.csv")
print("La consulta utiliza los resultados ya preparados, sin nuevas descargas.")


Pronóstico guardado: outside_forecast_horizon
Selector: E:\Programming\Master\Proyecto_TFM\app\index.html
Catálogo: E:\Programming\Master\Proyecto_TFM\data\gold\planning_eclipse_catalog.csv
La consulta utiliza los resultados ya preparados, sin nuevas descargas.
